# Divergence comparison on real data -- one held time point

The same psi comparison as `divergence_outliers`, on the real leave-one-out benchmark: hold out one
time point (embryoid day 13.5 / statefate day 4.0), fit the leave-one-out estimator at that time
with each psi (KL / chi^2 / Hellinger / TV) plus balanced OT, and score MMD / EMD / W2 against the
true held-out cells -- the same metrics and setup as the main LOO benchmark, so the rows line up.

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO_ROOT = P.REPO
import uotreg as U
from uotreg.config import ModelConfig, TrainConfig, UOTConfig
from uotreg.metrics import mmd_rbf, emd, w2, _HAS_POT
from uotreg.divergences import available_divergences
from uotreg import datasets

print("uotreg", U.__version__, "| psi:", available_divergences(),
      "| EMD/W2:", "EXACT (POT)" if _HAS_POT else "*** SLICED PROXY -- pip install pot ***")

## Parameters
`DATASET` picks embryoid or statefate (run once each). `TAU` is shared by all psi;
`TRAIN_OUTER` is the per-psi training budget.

In [ ]:
DATASET     = "embryoid"             # "embryoid" | "statefate"
DIM         = 20                     # 10 | 20 (50 needs h5py/scanpy)
DEVICE      = "auto"                 # "auto" | "cpu" | "cuda"
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE       = 1
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = 0

INCLUDE_BAD = True                   # also fit Hellinger + TV (negative controls)
DIVERGENCES = ["kl", "chi2"] + (["hellinger", "tv"] if INCLUDE_BAD else [])
ADD_BALANCED = True

# per-dataset config, mirroring the main LOO benchmark: held day, tau, kernel bandwidth h
# (None -> Silverman), gaussian init scale, per-psi budget, and the saved d=20 data-init.
_DEF = {"embryoid": dict(held=13.5, tau=5.0, h=3.0, gscale=10.0, outer=40,
                         ini20="data/ini/G_embryoid20_256_Day4_ini.pth"),
        "statefate": dict(held=4.0, tau=1.0, h=None, gscale=10.0, outer=50,
                          ini20="data/ini/G_statefate20_256_Dayall_ini.pth")}[DATASET]
HELD        = _DEF["held"]           # the single held-out time point
TAU         = _DEF["tau"]            # shared unbalanced tolerance
H_BENCH     = _DEF["h"]              # kernel bandwidth (None -> Silverman)
TRAIN_OUTER = 6 if SMOKE else _DEF["outer"]  # per-psi outer-iters R
BENCH_REPEATS = 3 if SMOKE else 30
BENCH_N     = 500 if SMOKE else 1000
N_PRED      = 2000

## Data + the held-out split
Every psi is fit on the SAME remaining snapshots and scored against the SAME held-out cells.

In [ ]:
ds = (datasets.load_embryoid(P.DATA_DIR, d=DIM) if DATASET == "embryoid" else
      datasets.load_statefate(P.DATA_DIR, d=DIM))
if SMOKE:
    ds = datasets.subsample(ds, n_per_time=1500, seed=0)
timepoints = ds.timepoints.tolist()
arrays = [np.asarray(a, np.float32) for a in ds.arrays]
j = timepoints.index(HELD)
assert 0 < j < len(timepoints) - 1, f"held {HELD} not interior in {timepoints}"
loo_times = [t for i, t in enumerate(timepoints) if i != j]
loo_samplers = [s for i, s in enumerate(U.samplers_from_arrays(arrays, device=DEVICE)) if i != j]
loo_pooled = U.TensorSampler(np.concatenate([arrays[i] for i in range(len(arrays)) if i != j], 0), device=DEVICE)
truth = arrays[j]


def silverman(times):
    X = np.asarray(times, float); return float(1.06 * np.std(X) * len(X) ** (-1 / 5))


h_b = silverman(loo_times) if H_BENCH is None else H_BENCH
print(f"{DATASET} d={DIM}: times={timepoints} | held Day{HELD} (idx {j}) | loo_times={loo_times} "
      f"| h={round(h_b,3)} tau={TAU} outer={TRAIN_OUTER}")

## Fit the leave-one-out estimator at the held day for each psi (+ balanced OT)
Same model / training / init for every run (a psi-agnostic shared init); only the divergence changes.

In [ ]:
model = ModelConfig(dim=DIM, latent_dim=DIM, gen_hidden=256, gen_layers=4, gen_dropout=0.05,
                    map_hidden=(256 if DATASET == "embryoid" else 196), map_layers=5,
                    pot_hidden=(256 if DATASET == "embryoid" else 196), pot_layers=5,
                    dropout=0.05, batchnorm=False)


def make_train():
    return TrainConfig(outer_iters=TRAIN_OUTER, d_iters=(10 if SMOKE else 50),
                       t_iters=(3 if SMOKE else 10), g_iters=(10 if SMOKE else 50), batch_size=64,
                       batch_size_g=128, lr_map=3e-4, lr_pot=3e-4, lr_gen=1e-4, weight_decay_td=1e-10,
                       weight_decay_gen=1e-8, device=DEVICE, verbose=False, seed=0)


# Initialization: at d=20 every psi starts from the SAME saved data-init generator (psi-agnostic,
# matching the main benchmark), so the comparison is fair; d=10/50 fall back to the gaussian init.
INI_PATH = P.aux_path(_DEF["ini20"])
USE_INI  = (DIM == 20) and (not SMOKE) and os.path.exists(INI_PATH)
print("init:", f"saved d=20 ini '{os.path.basename(INI_PATH)}' (shared by every psi)" if USE_INI
      else f"gaussian (scale {_DEF['gscale']}, iters {600 if SMOKE else 10000})")


def fit_psi(relaxation, divergence):
    uc = UOTConfig(relaxation=relaxation, divergence=divergence, tau=TAU)
    e = U.DistributionEstimator(model=model, uot=uc, train=make_train())
    if USE_INI:
        fit_kw = dict(pretrained_generator=INI_PATH)   # same shared warm start for every psi (fair)
    else:
        fit_kw = dict(init="gaussian", init_data_sampler=loo_pooled,
                      init_kwargs={"iters": (600 if SMOKE else 10000), "gaussian_scale": _DEF["gscale"]})
    e.fit(loo_samplers, loo_times, query_time=HELD, h=h_b, weight_scheme="positive", threshold=0.01, **fit_kw)
    return e.sample(N_PRED), e


runs = ([("balanced OT", "balanced", "kl")] if ADD_BALANCED else []) \
    + [(f"UOT {dv}", "one-sided", dv) for dv in DIVERGENCES]
preds, estimators = {}, {}                 # estimators kept so the fitted generator can be saved
for label, relaxation, dv in runs:
    t0 = time.time()
    preds[label], estimators[label] = fit_psi(relaxation, dv)
    print(f"  fit {label:14s} in {time.time()-t0:5.0f}s")

# naive baselines (same as realdata_dist_est_dims)
rng = np.random.default_rng(0)
b, a = arrays[j - 1], arrays[j + 1]
preds["naive-midpoint"] = 0.5 * (b[rng.integers(0, len(b), N_PRED)] + a[rng.integers(0, len(a), N_PRED)])
preds["carry-forward"] = b[rng.integers(0, len(b), N_PRED)]

## Benchmark: MMD / EMD / W2 vs the true held-out cells (lower = better)

In [ ]:
def cell_sampler(cells):
    cells = np.asarray(cells)
    return lambda n: cells[np.random.default_rng().integers(0, len(cells), n)]


metric_fns = {"MMD": mmd_rbf, "EMD": emd, "W2": w2}
st = cell_sampler(truth)
res = {m: {k: [] for k in metric_fns} for m in preds}
for _ in range(BENCH_REPEATS):
    ref = st(BENCH_N)
    for m, g in preds.items():
        s = cell_sampler(g)(BENCH_N)
        for k, fn in metric_fns.items():
            res[m][k].append(fn(s, ref))
if SMOKE:
    print("\n  !! SMOKE=1: tiny budget -- numbers are a plumbing check only. SMOKE=0 for real.")
print(f"\n[{DATASET} d={DIM}] held Day{HELD} psi comparison ({BENCH_REPEATS} reps, N={BENCH_N}; lower=better):")
print(f"  {'method':16s}" + "".join(f"{k:>16s}" for k in metric_fns))
for m in preds:
    print(f"  {m:16s}" + "".join(f"{np.mean(res[m][k]):8.3f}+/-{np.std(res[m][k]):<5.3f}" for k in metric_fns))

## Save: predicted clouds, metrics, and the fitted generators -> `new_results/divergence`

In [ ]:
import json
RESULTS_W = P.results("divergence", write=True)
_tag = f"realdata_{DATASET}{DIM}_Day{HELD}"          # also used by the figure cell below
if SAVE:
    os.makedirs(RESULTS_W, exist_ok=True)
    np.savez(os.path.join(RESULTS_W, f"{_tag}_preds.npz"), truth=np.asarray(truth, np.float32),
             **{m.replace(" ", "_"): np.asarray(g, np.float32) for m, g in preds.items()})
    with open(os.path.join(RESULTS_W, f"{_tag}_metrics.json"), "w") as f:
        json.dump({m: {k: {"mean": float(np.mean(res[m][k])), "std": float(np.std(res[m][k]))} for k in metric_fns}
                   for m in preds}, f, indent=2)
    for label, est in estimators.items():
        est.save(os.path.join(RESULTS_W, f"G_{_tag}_{label.replace(' ', '')}.pth"))
    print(f"saved preds/metrics/{len(estimators)} generators -> {RESULTS_W}")
else:
    print("SAVE=0 -- results kept in memory (set SAVE=1 to write new_results/)")

## Figure: predicted vs true held-out cells (2x3)

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})
METHODS = [m for m in preds if m not in ("naive-midpoint", "carry-forward")]     # drop the two naive
_vrng = np.random.default_rng(0)
true_sub = np.asarray(truth)[_vrng.integers(0, len(truth), min(int(1.2 * N_PRED), len(truth)))]
fpanels = [("true held-out", None)] + [(m, np.asarray(preds[m])) for m in METHODS]
fig, axes = plt.subplots(2, 3, figsize=(12, 7.6), dpi=140, sharex=True, sharey=True)
for i, ax in enumerate(axes.ravel()):
    if i >= len(fpanels):
        ax.axis("off"); continue
    nm, g = fpanels[i]
    ax.scatter(true_sub[:, 0], true_sub[:, 1], s=6, c="0.72", alpha=0.45)
    if g is not None:
        ax.scatter(g[:, 0], g[:, 1], s=6, c="tab:purple", alpha=0.55)
        ax.set_title(f"{nm}  (W2 {np.mean(res[nm]['W2']):.2f})", fontsize=11)
    else:
        ax.set_title("true held-out", fontsize=11)
for ax in axes[:, 0]:
    ax.set_ylabel("PC2")
for ax in axes[-1, :]:
    ax.set_xlabel("PC1")
fig.suptitle(f"{DATASET} d={DIM}, held Day {HELD}: predicted vs true held-out cells (grey)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()